Загрузка конфиденциальных настроек

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


db_url = os.getenv("db_url") 
api_key = os.getenv("api_key")

Шаг 1. Импортирую необходимые библиотеки


In [ ]:
import pandas as pd 
import requests 
import sqlalchemy 
import psycopg2


Шаг 2. Загружаю данные по велопарковкам

In [ ]:

dataset_id = '916' 

all_data = [] # Список для накопления всех скачанных строк
top = 1000    # Максимально разрешенное API количество строк за раз
skip = 0      # Начальное смещение



while True:
    # Формируем ссылку с текущими параметрами пагинации
    url = f"https://apidata.mos.ru/v1/datasets/{dataset_id}/rows?api_key={api_key}&$top={top}&$skip={skip}"
    
    # Отправляем запрос
    response = requests.get(url)
    
    # Проверяем успешность запроса
    if response.status_code == 200:
        chunk = response.json()
        
        # Если сервер вернул пустой список, значит данные закончились
        if not chunk:
            break
            
        # Добавляем новую порцию данных в общий список
        all_data.extend(chunk)
        print(f"Загружено {len(all_data)} строк...")
        
        # Если пришло меньше 1000 строк, значит это последняя "страница"
        if len(chunk) < top:
            break
            
        # Увеличиваем смещение для следующего запроса на 1000
        skip += top
    else:
        print(f"Ошибка запроса: {response.status_code}")
        print(response.text)
        break

# создаем таблицу
if all_data:
    # Превращаем объединенный JSON в плоскую таблицу Pandas
    df = pd.json_normalize(all_data)
    
    print("\nДанные успешно загружены! Итоговый размер таблицы:", df.shape)
    
    
    display(df.head())
else:
    print("Данные не были загружены.")

Шаг 3. Очищаю данные

In [ ]:
# Очищаем названия колонок 
df.columns = df.columns.str.replace('Cells.', '')

# Удаляем лишние технические колонки
df = df.drop(columns=['global_id', 'Number'], errors='ignore')

# удаляем дубликаты строк, если они случайно затесались
df = df.drop_duplicates()

# Выводим количество пропусков по каждому столбцу
display(df.isna().sum())


display(df.head(3))

Шаг 4. Отправляю данные в PostgreSQL


In [ ]:
from sqlalchemy import create_engine

# Задаем строку подключения к БД
engine = create_engine(db_url)

# Отправляем очищенный DataFrame  в PostgreSQL
df.to_sql(
    name='moscow_bike_parking', 
    con=engine,             
    if_exists='replace',    
)
index=False             